In [ ]:
import warnings
import sys
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())
warnings.filterwarnings('ignore')
sys.path.append('../..')

In [ ]:
from typing import List
from langchain_core.documents import Document

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from vector_store import VectorStoreHelper


vector_store_helper = VectorStoreHelper(search_index_name="rag")
filter_vector_opt = [{"type": "filter", "path": "source"}]
vector_store_helper.create_vector_search_index(
    search_index_name="rag", filter=filter_vector_opt)
vector_store = vector_store_helper.get_vector_store()
retriever = vector_store.as_retriever(search_type="mmr",
                                      search_kwargs={"k": 3, "fetch_k": 5})

In [ ]:
import bs4
from preprocess import TextProcessor

def load_docs() -> List[Document]:
    urls = [
        "https://lilianweng.github.io/posts/2023-06-23-agent/",
        "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
        "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
    ]

    loader = [WebBaseLoader(web_paths=(url,), bs_kwargs=dict(parse_only=bs4.SoupStrainer(
        class_=("post-content", "post-title", "post-header")))) for url in urls]

    docs = [d.load() for d in loader]
    docs_list = [item for sublist in docs for item in sublist]
    text_preprocess = TextProcessor()
    for doc in docs_list:
        doc.page_content = text_preprocess.preprocessing(doc.page_content)

    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=200, chunk_overlap=10)
    splits = text_splitter.split_documents(docs_list)

    return splits

# vector_store_helper.clear_data()
# vector_store_helper.add_data(doc=load_docs())

In [ ]:
def format_docs(docs: List[Document]):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
# Retrieval Grader

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings

from pydantic import BaseModel, Field


class GradeDocuments(BaseModel):
    """Binary score for relevance check on retriever documents."""

    binary_score: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no'")


llm = ChatNVIDIA(
    model="meta/llama-3.1-405b-instruct",
    api_key=os.environ["NVIDIA_API_KEY"],
    temperature=0.0,
)
structured_llm_grader = llm.with_structured_output(GradeDocuments)

system_prompt = """You are a grader assessing relevance of a retrieved document to a user question. \n 
    It does not need to be a stringent test. The goal is to filter out erroneous retrievals. \n
    If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. \n
    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question."""
human_prompt = "Retrieved document: \n\n {document} \n\n User question: {question}"
grade_prompt = ChatPromptTemplate.from_messages([("system", system_prompt), ("human", human_prompt)])
retrieval_grader = grade_prompt | structured_llm_grader

In [ ]:
### Generate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatNVIDIA(
    model="meta/llama-3.1-405b-instruct",
    temperature=0.5,
)

rag_prompt = """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\n
          Question: {question}\n
          Context: {context}\n
          Answer: """
generate_prompt = ChatPromptTemplate.from_template(rag_prompt)
rag_chain = generate_prompt | llm | StrOutputParser()

In [ ]:
# Hallucination Grader

class GradeHallucinations(BaseModel):
    """Binary score for hallucination present in generation answer."""

    binary_score: str = Field(
        description="Answer is grounded in the facts, 'yes' or 'no'")


llm = ChatNVIDIA(
    model="meta/llama-3.1-70b-instruct",
    temperature=0.0,
)
structured_llm_grader = llm.with_structured_output(GradeHallucinations)

system_prompt = """You are a grader assessing whether an LLM generation is grounded in / supported by a set of retrieved facts. \n 
     Give a binary score 'yes' or 'no'. 'Yes' means that the answer is grounded in / supported by the set of facts."""
human_prompt = "Set of facts: \n\n {documents} \n\n LLM generation: {generation}"
hallucination_prompt = ChatPromptTemplate.from_messages(
    [("system", system_prompt), ("human", human_prompt)])
hallucination_grader = hallucination_prompt | structured_llm_grader

In [ ]:
# Answer Grader

class GradeAnswer(BaseModel):
    """Binary score to assess answer addresses question."""

    binary_score: str = Field(
        description="Answer addresses the question, 'yes' or 'no'")


llm = ChatNVIDIA(
    model="meta/llama-3.1-70b-instruct",
    temperature=0.0,
)
structured_llm_grader = llm.with_structured_output(GradeAnswer)

system_prompt = """You are a grader assessing whether an answer addresses / resolves a question \n 
     Give a binary score 'yes' or 'no'. 'Yes' means that the answer resolves the question."""
human_prompt = "User question: \n\n {question} \n\n LLM generation: {generation}"
answer_prompt = ChatPromptTemplate.from_messages(
    ["system", system_prompt, ("human", human_prompt)])
answer_grader = answer_prompt | structured_llm_grader

In [ ]:
### Question Re-writer

llm = ChatNVIDIA(
    model="meta/llama-3.1-70b-instruct",
    temperature=0.5,
)

system_prompt = """You a question re-writer that converts an input question to a better version that is optimized \n 
     for vectorstore retrieval. Look at the input and try to reason about the underlying semantic intent / meaning."""
re_write_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        (
            "human",
            "Here is the initial question: \n\n {question} \n Formulate an improved question.",
        ),
    ]
)
question_rewriter = re_write_prompt | llm | StrOutputParser()

In [ ]:
### Graph state

from typing import List
from typing_extensions import TypedDict

class State(TypedDict):
    """
    Represents the state of our graph.

    Attributes:
        question: question
        generation: LLM generation
        documents: list of documents
    """
    
    question: str
    generation: str
    documents: List[Document]


In [ ]:
### Nodes function

def retrieve(state: State) -> State:
    """
    Retrieve documents

    Args:
        state (schemas.State): The current graph state

    Returns:
        state (schemas.State): New key added to state, documents, that contains retrieved documents
    """
    question = state["question"]
    documents = retriever.invoke(question)
    return {"documents": documents, "question": question}

def generate(state: State) -> State:
    """
    Generate answer

    Args:
        state (schemas.State): The current graph state

    Returns:
        state (schemas.State): New key added to state, generation, that contains LLM generation
    """
    question = state["question"]
    docs = state["documents"]

    generation = rag_chain.invoke({"context": format_docs(docs), "question": question})
    return {"documents": docs, "question": question, "generation": generation}

def grade_documents(state: State) -> State:
    """
    Determines whether the retrieved documents are relevant to the question.

    Args:
        state (schemas.State): The current graph state

    Returns:
        state (schemas.State): Updates documents key with only filtered relevant documents
    """
    question = state["question"]
    docs = state["documents"]
    filtered_docs = []
    for d in docs:
        score = retrieval_grader.invoke({"question": question, "document": d.page_content})
        grade = score.binary_score
        if grade == "yes":
            filtered_docs.append(d)
        else:
            continue
    return {"documents": filtered_docs, "question": question}

def transform_query(state: State) -> State:
    """
    Transform the query to produce a better question.

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): Updates question key with a re-phrased question
    """
    question = state["question"]
    docs = state["documents"]

    optimize_question = question_rewriter.invoke({"question": question})
    return {"documents": docs, "question": optimize_question}

In [ ]:
### Edges

def decide_to_generate(state: State):
    """
    Determines whether to generate an answer, or re-generate a question.

    Args:
        state (dict): The current graph state

    Returns:
        str: Binary decision for next node to call
    """
    filtered_docs = state["documents"]
    if not filtered_docs:
        # All document not satisfied 
        return "transform_query"
    else:
        return "generate"
    

def grade_generation_v_documents_and_question(state: State):
    """
    Determines whether the generation is grounded in the document and answers question.

    Args:
        state (dict): The current graph state

    Returns:
        str: Decision for next node to call
    """
    question = state["question"]
    documents = state["documents"]
    generation = state["generation"]

    score = hallucination_grader.invoke({"documents": format_docs(documents), "generation": generation})
    grade = score.binary_score

    if grade == "yes":
        score = answer_grader.invoke({"question": question, "generation": generation})
        grade = score.binary_score
        if grade == "yes":
            return "useful"
        else: 
            return "not useful"
    else:
        return "not supported"

In [ ]:
from langgraph.graph import END, StateGraph, START

workflow = StateGraph(State)

workflow.add_node("retrieve", retrieve)
workflow.add_node("generate", generate)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("transform_query", transform_query)

workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges(
    "grade_documents",
    decide_to_generate,
    {
        "transform_query": "transform_query",
        "generate": "generate",
    }
)
workflow.add_edge("transform_query", "retrieve")
workflow.add_conditional_edges(
    "generate",
    grade_generation_v_documents_and_question,
    {
        "useful": END,
        "not useful": "generate",
        "not supported": "transform_query"
    }
)

In [ ]:
from pprint import pprint

def pretty_print_stream_chunk(chunk):
    for node, value in chunk.items():
        if "generation" in value:
            pprint(value["generation"])

In [ ]:
from memory_saver import MongoDBSaver

with MongoDBSaver.from_conn_info(host=os.environ.get("MONGODB_ATLAS_CLUSTER_URI"), db_name="langchain_test_db") as checkpoint:
    graph = workflow.compile(checkpointer=checkpoint)
    config = {"configurable": {"user_id": "1", "thread_id": "chat_thread_1"}}
    while True:
        query = input("Enter messages:")
        if query == 'stop':
            break
        for chunk in graph.stream({"question": query}, config=config):
            pretty_print_stream_chunk(chunk)

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))